# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [14]:
# Importation des packages nécessaires

import pandas as pd
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [15]:
# Chargement des données du mapping
df_mapping_initial = pd.read_csv("../data_finale/mapping_worldfootballR/mapping_fbref_tm.csv", encoding='latin1')

# Chargement des données du dataset Soccerdata
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

# Chargement des données du dataset Transfermarkt
df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

# Préparation des données de Transfermarkt
df_tm_initial = prepare_transfermarkt_data(
    df_players,
    df_valuations
)

# Chargement des données du dataset de blessures Transfermarkt
df_blessures = pd.read_csv("../data/dataset_blessures.csv")

# Préparation des données de blessures Transfermarkt
df_blessures_initial = aggregate_injuries_by_season(df_blessures)

Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [16]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm, df_blessures = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial, df_blessures_initial)

## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [17]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm, df_blessures)

[1] Nom exact (mapping)     : 17067 | restants : 833
[2] Fuzzy nom (mapping)     :   124 | restants : 709


In [28]:
# 1. Filtrer uniquement les joueurs matchés via la méthode Fuzzy
df_fuzzy_matches = df_final[df_final['match_method'].str.startswith('fuzzy_name', na=False)].copy()

# 2. Extraire le score numérique de la chaîne 'fuzzy_name(XX.X)' pour pouvoir trier
df_fuzzy_matches['fuzzy_score'] = df_fuzzy_matches['match_method'].str.extract(r'\((.*?)\)').astype(float)

# 3. Sélectionner et ordonner les colonnes pour une inspection visuelle confortable
# (Ajustez 'player_name' ou 'team' selon les vrais noms de colonnes de votre df_soccerdata)
cols_to_inspect = [
    'join_key',         # Le nom d'origine (SoccerData / FBref)
    'player',             # Le nom récupéré de Transfermarkt (df_tm)
    'fuzzy_score',      # Le score de similarité obtenu
    'season_year',      # La saison du match
    'tm_id'             # L'identifiant Transfermarkt associé
]

# Optionnel : Ajouter le club ou championnat si présent dans votre df_soccerdata (ex: 'team')
if 'team' in df_fuzzy_matches.columns:
    cols_to_inspect.insert(2, 'team')

# 4. Afficher les résultats triés du score le plus bas au plus haut 
# (C'est en bas de tableau que se cachent les potentielles erreurs de matching)
df_inspection = df_fuzzy_matches[cols_to_inspect].sort_values(by='fuzzy_score', ascending=True)

print("\nTop des matchs avec les scores les plus hauts")
print(df_inspection.tail(20).to_string(index=False))


Top des matchs avec les scores les plus hauts
            join_key               player            team  fuzzy_score  season_year     tm_id
     karl etta eyong      Karl Etta Eyong      Villarreal        100.0         2025 1038950.0
     karl etta eyong      Karl Etta Eyong         Levante        100.0         2025 1038950.0
         marc pubill          Marc Pubill Atlético Madrid        100.0         2025  844637.0
vladyslav krapyvtsov Vladyslav Krapyvtsov          Girona        100.0         2025 1169436.0
       jonathan rowe        Jonathan Rowe       Marseille        100.0         2025  579346.0
    hakon haraldsson     Hákon Haraldsson           Lille        100.0         2025  652275.0
       idrissa gueye        Idrissa Guèye            Metz        100.0         2025  126665.0
    merveille papela     Merveille Papela        Mainz 05        100.0         2020  405689.0
           keke topp            Keke Topp   Werder Bremen        100.0         2024  701757.0
      cassian

In [27]:
print("\nTop des matchs avec les scores les plus bas")
print(df_inspection.head(20).to_string(index=False))


Top des matchs avec les scores les plus bas
                    join_key                  player            team  fuzzy_score  season_year    tm_id
                  joe knight              Joe Knight        Brighton         85.7         2025 425026.0
              toni fernandez          Toni Fernández       Barcelona         85.7         2025 467271.0
              andres antanon          Andrés Antañón      Celta Vigo         85.7         2025 993722.0
        nikola krstovi u0107         Nikola Krstović        Atalanta         85.7         2025 124715.0
         bartosz bia u0142ek          Bartosz Białek       Wolfsburg         85.7         2020  24496.0
    vasilije ad u017ei u0107          Vasilije Adžić        Juventus         85.7         2025 423609.0
               antonio arena           Antonio Arena            Roma         85.7         2025 241889.0
          giuseppe ambrosino      Giuseppe Ambrosino          Napoli         85.7         2025 579746.0
    vasilije ad u01

Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [19]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')
still_missing.to_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', index=False, sep=',', encoding='utf-8-sig')
df_soccerdata.to_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

## **Les joueurs orphelins**

In [21]:
still_missing = pd.read_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', encoding='utf-8-sig')
df_soccerdata = pd.read_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', encoding='utf-8-sig')

In [22]:
nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 643
Taux joueurs orphelins : 10.38%


In [ ]:
orphelins_par_saison = (
    still_missing
    .groupby('season_year')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season
2021      8
2122     26
2223     42
2324     72
2425     38
2526    523
dtype: int64
